Importing necessary libraries

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from urllib.parse import quote_plus
from sqlalchemy import create_engine
from matplotlib import pyplot as plt

Data_Loading

In [ ]:
df = pd.read_csv(r"C:\Users\Salman\Data science\Luxury_Housing_analysis\Luxury_Housing_Bangalore.csv")
display(df.head(5))
df.shape

Data cleaning and preprocessing

In [ ]:
df.info()

In [ ]:
for col in df:
    print(col)
    print(df[col].value_counts())

In [ ]:
df['Ticket_Price_Cr'] = df['Ticket_Price_Cr'].str.replace("Cr", "").str.replace("₹", "").str.strip().astype(float)

In [ ]:
df['Purchase_Quarter'] = pd.to_datetime(df['Purchase_Quarter'])

In [ ]:
df['Micro_Market'].unique()

In [ ]:
df['Micro_Market'] = df['Micro_Market'].str.strip().str.title()

In [ ]:
df['Configuration'] = df['Configuration'].replace({
    '3Bhk' : '3BHK',
    '3bhk' : '3BHK',
    '4bhk' : '4BHK',
    '4Bhk' : '4BHK',
    '5bhk+' : '5BHK+',
    '5Bhk+' : '5BHK+'
})

In [ ]:
df['Property_ID'].duplicated().sum()

In [ ]:
df = df.drop_duplicates(subset='Property_ID').reset_index(drop=True)

Function to find outliers

In [ ]:
def outlier(df):
    Q1 = df.quantile(0.25)
    Q3 = df.quantile(0.75)
    IQR = Q3 - Q1
    upper_limit = Q3 + IQR
    lower_limit = Q1 - IQR
    return upper_limit, lower_limit
outlier(df['Unit_Size_Sqft'])

In [ ]:
outlier(df['Ticket_Price_Cr'])

In [ ]:
len(df[(df['Ticket_Price_Cr']<5.29) | (df['Ticket_Price_Cr']>18.18)])

In [ ]:
len(df[(df['Unit_Size_Sqft']<1459) | (df['Unit_Size_Sqft']>10516)])

In [ ]:
df[df['Unit_Size_Sqft']==-1]

In [ ]:
df[(df['Ticket_Price_Cr']<1)]

Filling null values and outliers

In [ ]:
df['Unit_Size_Sqft'] = df['Unit_Size_Sqft'].replace(-1.0, np.nan)
df['Unit_Size_Sqft'] = df['Unit_Size_Sqft'].fillna(df.groupby(['Developer_Name','Configuration'])['Unit_Size_Sqft'].transform('mean'))

In [ ]:
df.loc[df['Ticket_Price_Cr'] < 0, 'Ticket_Price_Cr'] = np.nan

In [ ]:
df.loc[df['Ticket_Price_Cr'] > 20, 'Ticket_Price_Cr'] = np.nan

In [ ]:
df['Ticket_Price_Cr'] = df['Ticket_Price_Cr'].fillna(df.groupby(['Unit_Size_Sqft','Configuration','Transaction_Type'])['Ticket_Price_Cr'].transform('mean'))
df['Ticket_Price_Cr'] = df['Ticket_Price_Cr'].fillna(
    df.groupby(['Configuration','Transaction_Type'])['Ticket_Price_Cr'].transform('mean')
)

In [ ]:
df['Amenity_Score'] = df['Amenity_Score'].fillna(
    df.groupby(['Developer_Name','Configuration'])['Amenity_Score'].transform('mean')
)

In [ ]:
df['Buyer_Comments'] = df['Buyer_Comments'].fillna(df['Buyer_Comments'].mode()[0])

Creating new columns (Price_per_Sqft, Quarter_Number, Booking_Flag)

In [ ]:
df['Price_per_Sqft'] = df['Ticket_Price_Cr'] * 10000000 / df['Unit_Size_Sqft']

In [ ]:
df['Quarter_Number'] = df['Purchase_Quarter'].dt.quarter

In [ ]:
df['Booking_Flag'] = 1

In [ ]:
df.info()

In [ ]:
df.describe()

Establishing database connection and transferring cleaned dataframe to SQL as Table

In [ ]:
user = "shaik"
password = "Password@000"
host = "localhost"
port = 3306
db = "luxury_housing_price_analysis"
autocommit = True
encoded_password = quote_plus(password)

engine = create_engine(
    f"mysql+pymysql://{user}:{encoded_password}@{host}:{port}/{db}"
)

def to_sql(dataframe, table_name):
    try:
        dataframe.to_sql(
            name=table_name,
            con=engine,
            index=False,
            if_exists='fail'
        )
        print("First run: Table created and data uploaded successfully!")

    except ValueError:
        print("Table already exists. Skipping upload to keep original data.")

In [ ]:
to_sql(df , 'luxury_housing_data')